# Transportability: Covariate Shift, Weighting, and Causal Effect Transport

**Purpose.** Assess whether and how models/effects estimated in source countries transport to a held-out target country (leave-one-country-out across 4 clients). We measure covariate shift, estimate importance weights, and compare source/weighted/target-truth ATEs using the project's semi-synthetic causal truth injection.

## Learning objectives
- Quantify covariate shift (SMD, KS) and overlap via a domain classifier.
- Estimate and diagnose transport weights $w(x)=\frac{P(T=1\mid x)}{P(T=0\mid x)}$ with clipping and effective sample size (ESS).
- Transport a causal ATE from source to target and compare against injected target truth.
- Evaluate sensitivity of transported estimates to weight clipping.

## Background and math

### Covariate shift and standardized mean difference (SMD)
For feature $x_j$,
$$
\mathrm{SMD}_j = \frac{|\bar x_{j,S}-\bar x_{j,T}|}{s_{p,j}}, \quad s_{p,j}^2=\frac{(n_S-1)s_{S,j}^2+(n_T-1)s_{T,j}^2}{n_S+n_T-2}.
$$
*Meaning here:* larger SMD indicates the source and target differ on that covariate, threatening transportability.

### Importance weights and ESS
A domain classifier estimates $e(x)=P(T=1\mid x)$ (target membership). Odds weights $w(x)=\frac{e(x)}{1-e(x)}$ (for source units) reweight source toward target. Effective sample size:
$$
\mathrm{ESS}=\frac{(\sum_i w_i)^2}{\sum_i w_i^2}.
$$
*Meaning here:* low ESS means few source samples dominate after reweighting (high variance/poor overlap).

### Transported ATE (covariate-shift adjustment)
Under conditional exchangeability given $X$, positivity/overlap, and no unmodeled effect modifiers outside $X$, an effect can be transported by reweighting source units to match target covariate distribution. We compare source ATE, weighted (transported) ATE, and target ground-truth ATE from injected semi-synthetic truth.
*Meaning here:* agreement with target truth supports transport assumptions in this synthetic setup; disagreement signals shift/violation.

## Assumptions and limitations
- Semi-synthetic truth is injected (`inject_truth`), so "truth" tests estimator/transport mechanics, not real-world causality.
- Weights rely on measured covariates and a correctly specified domain model; hidden effect modifiers or positivity violations break transport.
- Results depend on clipping and sample size (light mode uses subsamples).


In [ ]:
import sys
from pathlib import Path
_here = Path.cwd().resolve()
for _p in [_here, _here / "notebooks", _here.parent]:
    if (_p / "_nb_common.py").exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
        break
import _nb_common as nb
nb.style()
rng = nb.set_seed()
nb.env_report()


In [ ]:
from dataclasses import dataclass
@dataclass(frozen=True)
class Cfg:
    seed: int = nb.SEED
    n_rows: int = 1500 if nb.MODE == "light" else 5000
    clip_q: float = 0.99
    n_boot: int = 80 if nb.MODE == "light" else 200
cfg = Cfg(); cfg


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from insurance_fl.causal.truth import inject_truth, ground_truth_ate

data_root = nb.require_data()
print('data_root:', data_root)
clients = nb.load_clients(split='train', n_max=cfg.n_rows)
assert clients, 'No client datasets loaded'
for k, v in clients.items():
    clients[k] = inject_truth(v.copy(), seed=nb.SEED, base_effect=5.0)
sample = next(iter(clients.values()))
TREAT = 'has_claims' if 'has_claims' in sample.columns else None
if TREAT is None:
    raise RuntimeError('Treatment column has_claims not found')
OUT_TRUE = 'claim_risk_score_true'
adj_set = [c for c in ['age','annual_income','savings_balance','credit_score','debt_to_income_ratio'] if c in sample.columns]
feat_cols = [
    c for c in sample.select_dtypes(include=[np.number]).columns
    if c not in ['customer_id', OUT_TRUE, 'effect_has_claims', 'claim_risk_score',
                 'customer_lifetime_value', 'will_renew', 'is_high_risk', 'customer_segment']
]
print('treatment:', TREAT, 'adj_set:', adj_set, 'n_features:', len(feat_cols))
assert len(adj_set) > 0, 'Expected adjustment covariates not found'
assert len(feat_cols) > 0, 'No numeric features available for domain/shift analysis'


## Helpers: shift, overlap, weights, ATE

In [ ]:
def smd_table(src, tgt, cols):
    rows = []
    for c in cols:
        s = pd.to_numeric(src[c], errors='coerce').dropna().values.astype(float)
        t = pd.to_numeric(tgt[c], errors='coerce').dropna().values.astype(float)
        if len(s) < 2 or len(t) < 2:
            continue
        sp = np.sqrt(((len(s) - 1) * np.var(s, ddof=1) + (len(t) - 1) * np.var(t, ddof=1)) / max(len(s) + len(t) - 2, 1))
        sp = sp if sp > 0 else 1e-12
        rows.append({'feature': c, 'smd': abs(np.mean(s) - np.mean(t)) / sp, 'ks_p': float(ks_2samp(s, t).pvalue)})
    return pd.DataFrame(rows).sort_values('smd', ascending=False)

# ATE helpers: prefer statsmodels for consistency; fallback to sklearn if unavailable
try:
    import statsmodels.api as sm
    _HAS_STATSMODELS = True
except Exception as _sm_err:
    sm = None  # type: ignore[assignment]
    _HAS_STATSMODELS = False
    from sklearn.linear_model import LinearRegression
    print(f"[fallback] statsmodels unavailable; using sklearn LinearRegression for ATE estimates ({_sm_err})")


def domain_weights(src, tgt, cols, seed=nb.SEED, clip_q=0.99):
    Xs = src[cols].values
    Xt = tgt[cols].values
    X = np.vstack([Xs, Xt])
    y = np.concatenate([np.zeros(len(Xs)), np.ones(len(Xt))])
    model = Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('sc', StandardScaler()),
        ('lr', LogisticRegression(max_iter=400, random_state=seed)),
    ])
    model.fit(X, y)
    p = model.predict_proba(Xs)[:, 1]
    p = np.clip(p, 1e-4, 1 - 1e-4)
    dom_auc = roc_auc_score(y, model.predict_proba(X)[:, 1]) if len(np.unique(y)) > 1 else np.nan
    w = p / (1 - p)
    cap = np.quantile(w, clip_q) if clip_q < 1.0 else np.max(w)
    w_clip = np.clip(w, None, cap)
    ess = float((w_clip.sum() ** 2) / (np.sum(w_clip ** 2))) if np.sum(w_clip ** 2) > 0 else float(len(w_clip))
    return w_clip, ess, float(dom_auc) if np.isfinite(dom_auc) else np.nan, p, float(cap)


def backdoor_ate(df, treat=TREAT, y='claim_risk_score', Z=None):
    # simple g-computation via linear regression (continuous outcome demo)
    Z = Z or adj_set
    d = df[[treat, y] + list(Z)].dropna().copy()
    design_cols = [treat] + list(Z)
    if _HAS_STATSMODELS:
        X = sm.add_constant(d[design_cols].astype(float), has_constant='add')
        model = sm.OLS(d[y].astype(float), X).fit()
        d1 = d.copy(); d1[treat] = 1.0
        d0 = d.copy(); d0[treat] = 0.0
        X1 = sm.add_constant(d1[design_cols].astype(float), has_constant='add')
        X0 = sm.add_constant(d0[design_cols].astype(float), has_constant='add')
        X1 = X1.reindex(columns=X.columns, fill_value=0.0)
        X0 = X0.reindex(columns=X.columns, fill_value=0.0)
        ate = float((model.predict(X1) - model.predict(X0)).mean())
        return ate
    # sklearn fallback (g-computation with linear outcome model)
    X = d[design_cols].astype(float)
    yy = d[y].astype(float)
    model = LinearRegression()
    model.fit(X, yy)
    X1 = X.copy(); X1[treat] = 1.0
    X0 = X.copy(); X0[treat] = 0.0
    return float((model.predict(X1) - model.predict(X0)).mean())


def weighted_backdoor_ate(src, w, treat=TREAT, y='claim_risk_score', Z=None):
    Z = Z or adj_set
    d = src[[treat, y] + list(Z)].dropna().copy()
    idx = d.index
    ww = pd.Series(w, index=src.index).loc[idx].values.astype(float)
    if np.sum(ww) <= 0:
        raise ValueError('Non-positive total weight')
    ww = ww / ww.mean()
    design_cols = [treat] + list(Z)
    if _HAS_STATSMODELS:
        X = sm.add_constant(d[design_cols].astype(float), has_constant='add')
        model = sm.WLS(d[y].astype(float), X, weights=ww).fit()
        d1 = d.copy(); d1[treat] = 1.0
        d0 = d.copy(); d0[treat] = 0.0
        X1 = sm.add_constant(d1[design_cols].astype(float), has_constant='add')
        X0 = sm.add_constant(d0[design_cols].astype(float), has_constant='add')
        X1 = X1.reindex(columns=X.columns, fill_value=0.0)
        X0 = X0.reindex(columns=X.columns, fill_value=0.0)
        ate = float((model.predict(X1) - model.predict(X0)).mean())
        return ate
    # sklearn fallback (weighted linear regression)
    X = d[design_cols].astype(float)
    yy = d[y].astype(float)
    model = LinearRegression()
    model.fit(X, yy, sample_weight=ww)
    X1 = X.copy(); X1[treat] = 1.0
    X0 = X.copy(); X0[treat] = 0.0
    return float((model.predict(X1) - model.predict(X0)).mean())


def naive_ate(df, treat=TREAT, y='claim_risk_score'):
    d = df[[treat, y]].dropna()
    return float(d.loc[d[treat] == 1, y].mean() - d.loc[d[treat] == 0, y].mean())


## Leave-one-country-out transport experiments

In [ ]:
# Ensure statsmodels-optional ATE helpers are current (handles stale kernel definitions)
try:
    import statsmodels.api as sm
    _HAS_STATSMODELS = True
except Exception as _sm_err:
    sm = None  # type: ignore[assignment]
    _HAS_STATSMODELS = False
    from sklearn.linear_model import LinearRegression
    print(f"[fallback] statsmodels unavailable; using sklearn LinearRegression for ATE estimates ({_sm_err})")


def backdoor_ate(df, treat=TREAT, y='claim_risk_score', Z=None):
    # simple g-computation via linear regression (continuous outcome demo)
    Z = Z or adj_set
    d = df[[treat, y] + list(Z)].dropna().copy()
    design_cols = [treat] + list(Z)
    if _HAS_STATSMODELS:
        X = sm.add_constant(d[design_cols].astype(float), has_constant='add')
        model = sm.OLS(d[y].astype(float), X).fit()
        d1 = d.copy(); d1[treat] = 1.0
        d0 = d.copy(); d0[treat] = 0.0
        X1 = sm.add_constant(d1[design_cols].astype(float), has_constant='add')
        X0 = sm.add_constant(d0[design_cols].astype(float), has_constant='add')
        X1 = X1.reindex(columns=X.columns, fill_value=0.0)
        X0 = X0.reindex(columns=X.columns, fill_value=0.0)
        ate = float((model.predict(X1) - model.predict(X0)).mean())
        return ate
    # sklearn fallback (g-computation with linear outcome model)
    from sklearn.linear_model import LinearRegression
    X = d[design_cols].astype(float)
    yy = d[y].astype(float)
    model = LinearRegression()
    model.fit(X, yy)
    X1 = X.copy(); X1[treat] = 1.0
    X0 = X.copy(); X0[treat] = 0.0
    return float((model.predict(X1) - model.predict(X0)).mean())


def weighted_backdoor_ate(src, w, treat=TREAT, y='claim_risk_score', Z=None):
    Z = Z or adj_set
    d = src[[treat, y] + list(Z)].dropna().copy()
    idx = d.index
    ww = pd.Series(w, index=src.index).loc[idx].values.astype(float)
    if np.sum(ww) <= 0:
        raise ValueError('Non-positive total weight')
    ww = ww / ww.mean()
    design_cols = [treat] + list(Z)
    if _HAS_STATSMODELS:
        X = sm.add_constant(d[design_cols].astype(float), has_constant='add')
        model = sm.WLS(d[y].astype(float), X, weights=ww).fit()
        d1 = d.copy(); d1[treat] = 1.0
        d0 = d.copy(); d0[treat] = 0.0
        X1 = sm.add_constant(d1[design_cols].astype(float), has_constant='add')
        X0 = sm.add_constant(d0[design_cols].astype(float), has_constant='add')
        X1 = X1.reindex(columns=X.columns, fill_value=0.0)
        X0 = X0.reindex(columns=X.columns, fill_value=0.0)
        ate = float((model.predict(X1) - model.predict(X0)).mean())
        return ate
    # sklearn fallback (weighted linear regression)
    from sklearn.linear_model import LinearRegression
    X = d[design_cols].astype(float)
    yy = d[y].astype(float)
    model = LinearRegression()
    model.fit(X, yy, sample_weight=ww)
    X1 = X.copy(); X1[treat] = 1.0
    X0 = X.copy(); X0[treat] = 0.0
    return float((model.predict(X1) - model.predict(X0)).mean())


records=[]
details={}
for target_c in nb.CLIENTS:
    src_cids = [c for c in nb.CLIENTS if c!=target_c]
    src = pd.concat([clients[c].assign(_client=c) for c in src_cids], ignore_index=True)
    tgt = clients[target_c].copy()
    smd = smd_table(src, tgt, feat_cols)
    w, ess, dom_auc, psel, cap = domain_weights(src, tgt, feat_cols, clip_q=cfg.clip_q)
    assert np.all(np.isfinite(w)) and np.all(w >= 0)
    assert 0.0 < ess <= float(len(src)) + 1e-6
    assert np.isnan(dom_auc) or (0.0 <= dom_auc <= 1.0)
    ate_src = backdoor_ate(src)
    ate_w = weighted_backdoor_ate(src, w)
    ate_naive = naive_ate(src)
    ate_tgt = backdoor_ate(tgt)
    truth_tgt = ground_truth_ate(tgt)
    # bootstrap CI for weighted estimate
    boots=[]
    rngb=np.random.default_rng(nb.SEED)
    n=len(src)
    for _ in range(cfg.n_boot):
        idx=rngb.integers(0,n,n)
        try:
            boots.append(weighted_backdoor_ate(src.iloc[idx].reset_index(drop=True), w[idx]))
        except Exception:
            pass
    ci = np.quantile(boots,[0.025,0.975]) if len(boots)>5 else [np.nan,np.nan]
    worst_smd = float(smd['smd'].iloc[0]) if not smd.empty else np.nan
    assert np.isnan(worst_smd) or worst_smd >= 0.0
    records.append({
        'target': target_c,
        'target_country': nb.CLIENT_LABELS.get(target_c,target_c),
        'domain_auc': dom_auc,
        'ess': ess,
        'ess_frac': ess/len(src) if len(src) else np.nan,
        'worst_smd': worst_smd,
        'ate_naive_src': ate_naive,
        'ate_backdoor_src': ate_src,
        'ate_weighted_transport': ate_w,
        'ate_backdoor_target': ate_tgt,
        'truth_target': float(truth_tgt),
        'ci_low': float(ci[0]) if np.isfinite(ci[0]) else np.nan,
        'ci_high': float(ci[1]) if np.isfinite(ci[1]) else np.nan,
        'n_boot_ok': int(len(boots)),
    })
    details[target_c]={'smd':smd,'w':w,'psel':psel,'src':src,'tgt':tgt,'cap':cap}
res = pd.DataFrame(records)
assert not res.empty
res


## Figures

In [ ]:
# Figure 1: density overlays for top-shift features per target (first 2 targets shown compactly)
fig, axes = plt.subplots(2,2, figsize=(11,7))
axes = axes.ravel()
for i, target_c in enumerate(nb.CLIENTS[:4]):
    ax = axes[i]
    smd = details[target_c]['smd']
    feats = smd['feature'].head(2).tolist() if not smd.empty else feat_cols[:1]
    f = feats[0]
    src = details[target_c]['src']; tgt = details[target_c]['tgt']
    ax.hist(pd.to_numeric(src[f],errors='coerce').dropna(), bins=30, alpha=0.4, density=True, label='source (3 countries)')
    ax.hist(pd.to_numeric(tgt[f],errors='coerce').dropna(), bins=30, alpha=0.4, density=True, label=f'target {nb.CLIENT_LABELS.get(target_c,target_c)}')
    ax.set_title(f'Top-shift feature: {f}'); ax.legend(fontsize=8)
fig.tight_layout(); nb.save_fig(fig, 'transport_density_overlays'); plt.show()


**How to read this / what it shows.** Overlaid distributions for a highly shifted feature illustrate covariate shift between pooled sources and each held-out target.

In [ ]:
# Figure 2: propensity-of-selection and weight distributions
fig, axes = plt.subplots(1,2, figsize=(11,4))
for target_c in nb.CLIENTS:
    axes[0].hist(details[target_c]['psel'], bins=30, alpha=0.35, density=True, label=nb.CLIENT_LABELS.get(target_c,target_c))
    axes[1].hist(np.log10(details[target_c]['w']+1e-12), bins=30, alpha=0.35, density=True, label=nb.CLIENT_LABELS.get(target_c,target_c))
axes[0].set_title('Domain classifier P(target|x) on source samples'); axes[0].set_xlabel('Propensity of selection'); axes[0].set_ylabel('Density'); axes[0].legend(fontsize=8)
axes[1].set_title('Transport weights (log10, clipped)'); axes[1].set_xlabel('log10 weight'); axes[1].set_ylabel('Density'); axes[1].legend(fontsize=8)
fig.tight_layout(); nb.save_fig(fig, 'transport_propensity_weights'); plt.show()


**How to read this / what it shows.** Left: overlap in predicted target membership; mass near 0 or 1 indicates weak overlap/positivity risks. Right: weight distributions; heavy tails imply unstable reweighting (mitigated by clipping).

In [ ]:
# Figure 3: ESS and domain AUC by target
fig, ax = plt.subplots(figsize=(7.5,4))
x=np.arange(len(res))
ax.bar(x-0.15, res['ess_frac'], width=0.3, label='ESS / n_source')
ax.bar(x+0.15, res['domain_auc'], width=0.3, label='Domain classifier AUC')
ax.set_xticks(x); ax.set_xticklabels(res['target_country'], rotation=15)
ax.set_ylim(0,1.05); ax.set_ylabel('Fraction / AUC'); ax.set_title('Overlap diagnostics by target country'); ax.legend()
fig.tight_layout(); nb.save_fig(fig, 'transport_ess_auc'); plt.show()


**How to read this / what it shows.** Higher domain AUC means source/target are easier to distinguish (greater shift). Lower ESS fraction indicates less effective information after weighting.

In [ ]:
# Figure 4: ATE estimates vs target truth with CI
fig, ax = plt.subplots(figsize=(8,4.5))
x=np.arange(len(res)); w=0.18
ax.bar(x-1.5*w, res['ate_naive_src'], width=w, label='Naive source')
ax.bar(x-0.5*w, res['ate_backdoor_src'], width=w, label='Backdoor source')
ax.bar(x+0.5*w, res['ate_weighted_transport'], width=w, label='Weighted transport')
ax.bar(x+1.5*w, res['truth_target'], width=w, label='Target truth (injected)')
# CI error bars on weighted
yerr = np.vstack([res['ate_weighted_transport']-res['ci_low'], res['ci_high']-res['ate_weighted_transport']])
ax.errorbar(x+0.5*w, res['ate_weighted_transport'], yerr=yerr, fmt='none', ecolor='black', capsize=3)
ax.set_xticks(x); ax.set_xticklabels(res['target_country']); ax.set_ylabel('ATE (claim_risk_score units)')
ax.set_title('ATE estimates vs injected target truth (leave-one-country-out)'); ax.legend(fontsize=8)
fig.tight_layout(); nb.save_fig(fig, 'transport_ate_comparison'); plt.show()


**How to read this / what it shows.** Compares naive and adjusted source ATEs, transported (weighted) ATE with bootstrap CI, and injected target truth. Closer weighted estimates to truth suggest covariate-shift transport helps under stated assumptions.

In [ ]:
# Ensure statsmodels-optional ATE helpers are current (handles stale kernel definitions)
try:
    import statsmodels.api as sm
    _HAS_STATSMODELS = True
except Exception as _sm_err:
    sm = None  # type: ignore[assignment]
    _HAS_STATSMODELS = False
    from sklearn.linear_model import LinearRegression
    print(f"[fallback] statsmodels unavailable; using sklearn LinearRegression for ATE estimates ({_sm_err})")


def backdoor_ate(df, treat=TREAT, y='claim_risk_score', Z=None):
    # simple g-computation via linear regression (continuous outcome demo)
    Z = Z or adj_set
    d = df[[treat, y] + list(Z)].dropna().copy()
    design_cols = [treat] + list(Z)
    if _HAS_STATSMODELS:
        X = sm.add_constant(d[design_cols].astype(float), has_constant='add')
        model = sm.OLS(d[y].astype(float), X).fit()
        d1 = d.copy(); d1[treat] = 1.0
        d0 = d.copy(); d0[treat] = 0.0
        X1 = sm.add_constant(d1[design_cols].astype(float), has_constant='add')
        X0 = sm.add_constant(d0[design_cols].astype(float), has_constant='add')
        X1 = X1.reindex(columns=X.columns, fill_value=0.0)
        X0 = X0.reindex(columns=X.columns, fill_value=0.0)
        ate = float((model.predict(X1) - model.predict(X0)).mean())
        return ate
    # sklearn fallback (g-computation with linear outcome model)
    from sklearn.linear_model import LinearRegression
    X = d[design_cols].astype(float)
    yy = d[y].astype(float)
    model = LinearRegression()
    model.fit(X, yy)
    X1 = X.copy(); X1[treat] = 1.0
    X0 = X.copy(); X0[treat] = 0.0
    return float((model.predict(X1) - model.predict(X0)).mean())


def weighted_backdoor_ate(src, w, treat=TREAT, y='claim_risk_score', Z=None):
    Z = Z or adj_set
    d = src[[treat, y] + list(Z)].dropna().copy()
    idx = d.index
    ww = pd.Series(w, index=src.index).loc[idx].values.astype(float)
    if np.sum(ww) <= 0:
        raise ValueError('Non-positive total weight')
    ww = ww / ww.mean()
    design_cols = [treat] + list(Z)
    if _HAS_STATSMODELS:
        X = sm.add_constant(d[design_cols].astype(float), has_constant='add')
        model = sm.WLS(d[y].astype(float), X, weights=ww).fit()
        d1 = d.copy(); d1[treat] = 1.0
        d0 = d.copy(); d0[treat] = 0.0
        X1 = sm.add_constant(d1[design_cols].astype(float), has_constant='add')
        X0 = sm.add_constant(d0[design_cols].astype(float), has_constant='add')
        X1 = X1.reindex(columns=X.columns, fill_value=0.0)
        X0 = X0.reindex(columns=X.columns, fill_value=0.0)
        ate = float((model.predict(X1) - model.predict(X0)).mean())
        return ate
    # sklearn fallback (weighted linear regression)
    from sklearn.linear_model import LinearRegression
    X = d[design_cols].astype(float)
    yy = d[y].astype(float)
    model = LinearRegression()
    model.fit(X, yy, sample_weight=ww)
    X1 = X.copy(); X1[treat] = 1.0
    X0 = X.copy(); X0[treat] = 0.0
    return float((model.predict(X1) - model.predict(X0)).mean())


# Figure 5: sensitivity of transported ATE to weight clipping quantile
clip_qs = [0.8,0.9,0.95,0.99,1.0]
sens_rows=[]
for target_c in nb.CLIENTS:
    src_cids = [c for c in nb.CLIENTS if c!=target_c]
    src = pd.concat([clients[c] for c in src_cids], ignore_index=True)
    tgt = clients[target_c]
    for cq in clip_qs:
        w, ess, dom_auc, psel, cap = domain_weights(src, tgt, feat_cols, clip_q=cq)
        try:
            ate_w = weighted_backdoor_ate(src, w)
        except Exception as e:
            ate_w = np.nan
            print(f'[fallback] weighted ATE failed for {target_c} clip_q={cq}: {e}')
        sens_rows.append({
            'target': nb.CLIENT_LABELS.get(target_c,target_c),
            'clip_q': cq,
            'ate_weighted': ate_w,
            'ess': ess,
        })
sens = pd.DataFrame(sens_rows)
fig, ax = plt.subplots(figsize=(7.5,4))
for t, g in sens.groupby('target'):
    ax.plot(g['clip_q'], g['ate_weighted'], marker='o', label=t)
ax.set_title('Sensitivity of transported ATE to weight clipping quantile')
ax.set_xlabel('Clipping quantile (1.0=no clip)')
ax.set_ylabel('Weighted ATE')
ax.legend()
fig.tight_layout()
nb.save_fig(fig, 'transport_clipping_sensitivity')
plt.show()
sens.pivot(index='clip_q', columns='target', values='ate_weighted')


**How to read this / what it shows.** Stability across clipping levels suggests robustness; large swings indicate sensitivity to extreme weights/overlap issues.

In [ ]:
# Figure 6: worst SMD features per target (bar chart)
fig, axes = plt.subplots(2,2, figsize=(11,7)); axes=axes.ravel()
for i,target_c in enumerate(nb.CLIENTS):
    smd = details[target_c]['smd'].head(8)
    axes[i].barh(smd['feature'][::-1], smd['smd'][::-1], color='#2a6f97')
    axes[i].set_title(f"Worst SMDs: target {nb.CLIENT_LABELS.get(target_c,target_c)}"); axes[i].set_xlabel('SMD')
fig.tight_layout(); nb.save_fig(fig, 'transport_worst_smd'); plt.show()


**How to read this / what it shows.** Features with largest standardized mean differences drive covariate shift; they are candidates for adjustment/diagnostics.

## Quantitative summary


In [ ]:
res_out = res.copy()
res_out['transport_error_vs_truth'] = res_out['ate_weighted_transport'] - res_out['truth_target']
res_out


## Sensitivity / ablation
- **Clipping:** trades bias for variance; inspect ESS and estimate stability.
- **Feature set:** including/excluding effect modifiers or colliders changes weights and transport validity.
- **Model choice:** domain classifier/ATE estimators affect estimates; light mode limits sample size.

## Limitations and failure cases
- Transport assumes conditional exchangeability, positivity, and stable mechanisms given $X$; violations (hidden modifiers, poor overlap) invalidate results.
- Injected truth is a synthetic benchmark, not evidence of real causal effects.
- KS tests and domain AUC are diagnostics, not proofs of transportability.

## Conclusions and further experiments
- Leave-one-country-out analysis quantified shift, overlap (ESS/domain AUC), and transported ATEs against injected truth.
- Weight clipping sensitivity highlights overlap/variance trade-offs.
- Further work: richer transport estimators (doubly robust transport), effect-modifier analysis, and validation on additional targets/tasks.

## References
- J. Pearl and E. Bareinboim. "External Validity: From Do-Calculus to Transportability Across Populations." *Statistical Science*, 2014.
- I. Degtiar and S. Rose. "A Review of Generalizability and Transportability." *Annual Review of Statistics and Its Application*, 2023.
- P. R. Rosenbaum and D. B. Rubin. "The Central Role of the Propensity Score in Observational Studies for Causal Effects." *Biometrika*, 1983.
